# Tutorial 04: Empirical Probability Explorations, Central Limit Theorem

& Conditional Probabilities

Mathematics for Machine Learning (Mathe3)

Thomas Manke  
2026-10-07

## 1. Environment Setup (5 min)

Run the environment check cell below to confirm Python 3, NumPy, SciPy,
Matplotlib, and Seaborn are available:

``` {python}
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Environment check passed.")
print(f"NumPy Version: {np.__version__}")
```

------------------------------------------------------------------------

## 2. Task 1: Fast Empirical CLT & Sample Statistics Warm-up (15–20 min)

In this warm-up exercise, we explore the **Central Limit Theorem (CLT)**
and empirical sample statistics by simulating cohort averages of
non-Gaussian physical measurements across increasing sample sizes $N$.

### 2.1 Team Assignments & Physical Problem Context

Each team works with a distinct non-Gaussian physical measurement:

-   **Team A (Hospital Emergency Triage Response Delays)**:
    -   *Individual Random Variable $X_i$*: Delay (in hours) before an
        individual emergency patient sees a specialist. Individual
        delays are heavily right-skewed
        ($\text{Exponential}(\lambda=1)$)—most patients are triaged
        quickly ($0.2$ to $0.5$ hours), but occasional emergency
        backlogs create long delays ($3+$ hours).
    -   *Sample Mean $\bar{X}_N = \frac{1}{N}\sum_{i=1}^N X_i$*: The
        **average triage waiting delay across a hospital shift cohort of
        $N$ patients**.
-   **Team B (ICU Patient Monitor Measurement Errors)**:
    -   *Individual Random Variable $X_i$*: Sensor error (in units) of
        an individual digital patient monitor reading around baseline
        calibration. Individual errors are uniformly distributed
        ($\text{Uniform}(0, 1)$)—any error value between $0$ and $1$ is
        equally likely.
    -   *Sample Mean $\bar{X}_N = \frac{1}{N}\sum_{i=1}^N X_i$*: The
        **average measurement error across $N$ repeated sensor
        readings**.

------------------------------------------------------------------------

### 2.2 Instructions for Team A & Team B

1.  Run the pre-written simulation code cell below for your assigned
    distribution across sample sizes $N \in \{1, 5, 30, 100\}$ ($10,000$
    simulation trials per $N$).
2.  For each sample size $N$, compute the empirical sample statistics of
    the cohort averages $\bar{X}_N$:
    -   **Sample Mean**:
        $\bar{x} = \frac{1}{M} \sum_{m=1}^M \bar{X}_N^{(m)}$
    -   **Sample Variance**:
        $s^2 = \frac{1}{M-1} \sum_{m=1}^M (\bar{X}_N^{(m)} - \bar{x})^2$
    -   **Sample Standard Deviation**: $s = \sqrt{s^2}$
    -   **Standard Error of the Mean (SEM)**:
        $\text{SEM} = \frac{s}{\sqrt{N}}$
3.  Observe how your distribution shape changes as $N$ increases from
    $1 \to 100$.

``` {python}
# Task 1 Code Framework: Run for your Team's Assigned Variable
np.random.seed(42)
num_experiments = 10000
N_list = [1, 5, 30, 100]

# --- UNCOMMENT YOUR TEAM'S ASSIGNED VARIABLE BELOW ---
# TEAM = 'A'  # Team A: Hospital Triage Delays (Exponential, lambda=1)
TEAM = 'B'    # Team B: ICU Monitor Sensor Errors (Uniform, 0 to 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=False)

for idx, N in enumerate(N_list):
    if TEAM == 'A':
        # Team A: Draw N Exponential triage delays per experiment
        samples = np.random.exponential(scale=1.0, size=(num_experiments, N))
        pop_mean, pop_std = 1.0, 1.0
        dist_name = "Emergency Triage Delays (Exponential)"
        color_choice = "darkcyan"
    else:
        # Team B: Draw N Uniform sensor errors per experiment
        samples = np.random.uniform(low=0.0, high=1.0, size=(num_experiments, N))
        pop_mean, pop_std = 0.5, np.sqrt(1/12)
        dist_name = "ICU Sensor Errors (Uniform)"
        color_choice = "darkorange"
        
    # Compute cohort sample mean X_bar for each experiment
    sample_means = np.mean(samples, axis=1)
    
    # Calculate empirical sample statistics
    emp_mean = np.mean(sample_means)
    emp_var = np.var(sample_means, ddof=1)
    emp_std = np.std(sample_means, ddof=1)
    theo_sem = pop_std / np.sqrt(N)
    
    # Plot normalized histogram and KDE density
    sns.histplot(sample_means, kde=True, ax=axes[idx], color=color_choice, stat='density')
    axes[idx].set_title(f"Cohort Size N = {N}\nMean: {emp_mean:.3f}, Std: {emp_std:.3f}\nTheo SEM: {theo_sem:.3f}", fontsize=8)
    axes[idx].set_xlabel("Cohort Mean $\\bar{X}_N$")

plt.suptitle(f"Team {TEAM} — CLT Convergence & SEM Reduction: {dist_name}", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
```

> **Key Takeaway: The Central Limit Theorem & SEM Shrinkage**
>
> Regardless of whether your starting individual distribution was
> heavily right-skewed (Team A) or completely flat/uniform (Team B), the
> distribution of cohort averages $\bar{X}_N$ converges to a **symmetric
> Gaussian bell curve**
> $\mathcal{N}\left(\mu, \frac{\sigma^2}{N}\right)$ as $N$ increases.
>
> Crucially, the standard deviation of cohort averages—known as the
> **Standard Error of the Mean
> ($\text{SEM} = \frac{\sigma}{\sqrt{N}}$)**—shrinks by a factor of
> $\frac{1}{\sqrt{N}}$. Averaging over larger patient cohorts or
> repeated sensor readings systematically reduces operational and
> measurement uncertainty!

------------------------------------------------------------------------

## 3. Task 2: Discrete Medical Contingency Table & Non-Invertible Conditional Probabilities (40–45 min)

In this core exercise, we analyze a real-world medical diagnostic
screening dataset of $N_{\text{total}} = 1,000$ hospital patients
evaluated for a targeted clinical condition.

### 3.1 Dataset Setup: Raw Patient Frequencies

The matrix below records raw patient counts ($N_{XY}$) across **Disease
Status $X$** ($0 = \text{Healthy}, 1 = \text{Diseased}$) and
**Diagnostic Test Result $Y$**
($0 = \text{Negative}, 1 = \text{Positive}$):

``` {python}
# Raw Patient Count Matrix N_XY (2x2 Frequencies Table)
# Columns: Disease Status X (0: Healthy, 1: Diseased)
# Rows: Diagnostic Test Result Y (0: Negative, 1: Positive)
raw_counts = np.array([
    [850,  10],  # Y=0 (Negative): 850 Healthy, 10 Diseased (False Negatives)
    [ 50,  90]   # Y=1 (Positive):  50 Healthy (False Positives), 90 Diseased (True Positives)
])

x_labels = ['Healthy (X=0)', 'Diseased (X=1)']
y_labels = ['Negative (Y=0)', 'Positive (Y=1)']
```

------------------------------------------------------------------------

### 3.2 Step 1: Compute the Joint Probability Matrix $P(X, Y)$ (Both Teams)

Normalize the raw counts matrix by dividing every cell by total patients
$N_{\text{total}} = 1,000$ to compute cell-wise joint probabilities
$P(X=x, Y=y)$:

``` {python}
# Step 1: Normalize total count matrix by N_total = 1000
N_total = np.sum(raw_counts)
P_XY = raw_counts / N_total

print("Cell-Wise Joint Probability Matrix P(X, Y):")
print(P_XY)
print(f"\nSum of all Joint Probabilities: {np.sum(P_XY):.2f}")
```

------------------------------------------------------------------------

### 3.3 Step 2: Team Specific Marginal & Conditional Analysis

#### Instructions for Team A (Column Direction & $P(Y \mid X)$):

1.  Compute **Column Marginal Probabilities $P(X)$** by summing down the
    columns: $$P(X=x) = \sum_{y} P(X=x, Y=y)$$
2.  Compute **Conditional Probabilities $P(Y \mid X)$** by dividing
    joint probabilities by column marginals:
    $$P(Y=y \mid X=x) = \frac{P(X=x, Y=y)}{P(X=x)}$$
3.  Compute the **Diagnostic Sensitivity**:
    $P(Y=\text{Positive} \mid X=\text{Diseased})$.
4.  Compute the **False Positive Rate**:
    $P(Y=\text{Positive} \mid X=\text{Healthy})$.

``` {python}
# --- TEAM A CODE CELL: Column Marginals P(X) & Conditional P(Y | X) ---
marg_X = np.sum(P_XY, axis=0) # Sum over Y (rows) -> P(X=0), P(X=1)
cond_Y_given_X = P_XY / marg_X  # Divide each column by its marginal P(X)

print("--- TEAM A RESULTS ---")
print(f"Disease Prevalence P(X=Diseased): {marg_X[1]:.3f} ({marg_X[1]*100:.1f}%)")
print(f"Healthy Fraction P(X=Healthy):   {marg_X[0]:.3f} ({marg_X[0]*100:.1f}%)")
print("\nConditional Matrix P(Y | X):")
print("                  Healthy (X=0)  Diseased (X=1)")
print(f"Test Neg (Y=0):   {cond_Y_given_X[0,0]:.3f}          {cond_Y_given_X[0,1]:.3f}")
print(f"Test Pos (Y=1):   {cond_Y_given_X[1,0]:.3f}          {cond_Y_given_X[1,1]:.3f}")

# Plot Team A Conditional Probabilities P(Y | X)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Healthy (X=0)', 'Diseased (X=1)'], cond_Y_given_X[1, :], color='darkcyan', alpha=0.7, width=0.4)
ax.set_title('Team A: P(Test Positive | Disease Status X)\n(Sensitivity = 90.0%)', fontsize=10, fontweight='bold')
ax.set_ylabel('Conditional Probability $P(Y=1 \\mid X)$', fontsize=9)
ax.set_ylim(0, 1.0)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')
plt.show()
```

------------------------------------------------------------------------

#### Instructions for Team B (Row Direction & $P(X \mid Y)$):

1.  Compute **Row Marginal Probabilities $P(Y)$** by summing across the
    rows: $$P(Y=y) = \sum_{x} P(X=x, Y=y)$$
2.  Compute **Conditional Probabilities $P(X \mid Y)$** by dividing
    joint probabilities by row marginals:
    $$P(X=x \mid Y=y) = \frac{P(X=x, Y=y)}{P(Y=y)}$$
3.  Compute the **Positive Predictive Value (PPV)**:
    $P(X=\text{Diseased} \mid Y=\text{Positive})$.
4.  Compute the **Negative Predictive Value (NPV)**:
    $P(X=\text{Healthy} \mid Y=\text{Negative})$.

``` {python}
# --- TEAM B CODE CELL: Row Marginals P(Y) & Conditional P(X | Y) ---
marg_Y = np.sum(P_XY, axis=1) # Sum over X (columns) -> P(Y=0), P(Y=1)
cond_X_given_Y = (P_XY.T / marg_Y).T  # Divide each row by its marginal P(Y)

print("--- TEAM B RESULTS ---")
print(f"Total Positive Test Fraction P(Y=Pos): {marg_Y[1]:.3f} ({marg_Y[1]*100:.1f}%)")
print(f"Total Negative Test Fraction P(Y=Neg): {marg_Y[0]:.3f} ({marg_Y[0]*100:.1f}%)")
print("\nConditional Matrix P(X | Y):")
print("                  Test Neg (Y=0)  Test Pos (Y=1)")
print(f"Healthy (X=0):    {cond_X_given_Y[0,0]:.3f}          {cond_X_given_Y[1,0]:.3f}")
print(f"Diseased (X=1):   {cond_X_given_Y[0,1]:.3f}          {cond_X_given_Y[1,1]:.3f}")

# Plot Team B Conditional Probabilities P(X | Y)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Test Neg (Y=0)', 'Test Pos (Y=1)'], cond_X_given_Y[:, 1], color='darkorange', alpha=0.7, width=0.4)
ax.set_title('Team B: P(Diseased | Test Result Y)\n(Positive Predictive Value = 64.3%)', fontsize=10, fontweight='bold')
ax.set_ylabel('Conditional Probability $P(X=1 \\mid Y)$', fontsize=9)
ax.set_ylim(0, 1.0)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')
plt.show()
```

------------------------------------------------------------------------

## 4. Team Presentation & Joint Reflection (15 min)

Compare Team A’s result vs. Team B’s result side-by-side:

-   **Team A Result**: Test Sensitivity
    $P(Y=\text{Positive} \mid X=\text{Diseased}) = \mathbf{90.0\%}$
-   **Team B Result**: Positive Predictive Value
    $P(X=\text{Diseased} \mid Y=\text{Positive}) = \mathbf{64.3\%}$

> **Critical Philosophical & Mathematical Lesson: Non-Invertibility of
> Conditioning**
>
> Why is a patient with a positive test result only $64.3\%$ likely to
> actually have the disease, even though the test correctly identifies
> $90.0\%$ of diseased patients?
>
> Because **disease prevalence is low ($P(X=1) = 10\%$)**! The false
> positive pool among healthy patients ($50$ people) is significant
> relative to the true positive pool ($90$ people).
>
> **Takeaway**: Conditional probabilities **CANNOT be blindly
> inverted**: $$P(A \mid B) \neq P(B \mid A)$$
>
> To invert a conditional probability, you MUST apply **Bayes’
> Theorem**: $$P(X \mid Y) = \frac{P(Y \mid X) \cdot P(X)}{P(Y)}$$